# ???????? ???????????? ?????? ?? transformers

??????? ???????????? ??? ??????? ? **Google Colab**. ?? ??????? ???????????? ?????? ????????????? ???????? ?? ???? `transformers`, ???????????? ??????? ? ????????? ????????? ? ????????? ???????.


## ??? ?????? ???????

1. ????????????? ??????????.
2. ????????? CSV ?? ?????????? ????? ??? ?? ?????? ??????.
3. ??????? ?????? ? ????? ???????.
4. ????? ?????? ?? `train / validation / test`.
5. ??????? ?????? `transformers` ????? `Trainer`.
6. ????????? ???? ? `model1.pt`, ?????????? `save_pretrained`, ??????????? ? JSON-??????.


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy requests safetensors


## ??????? ? ??????? ????????????


In [ ]:
from __future__ import annotations

import json
import random
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import torch
from datasets import Dataset, DatasetDict
from google.colab import files
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

RANDOM_STATE = 42
MODEL_NAME = 'cointegrated/rubert-tiny2'
TEXT_COLUMN_CANDIDATES = ('text', 'content', 'title', 'description', '?????')
LABEL_COLUMN_CANDIDATES = ('label', 'category', 'topic', '?????', '?????????')
PROJECT_ROOT = Path('/content/isnews_transformers_project')
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
SPLIT_DATA_DIR = PROJECT_ROOT / 'data' / 'splits'
MODELS_DIR = PROJECT_ROOT / 'models'
MODEL_DIR = MODELS_DIR / 'transformer_model1'
TOKENIZER_DIR = MODELS_DIR / 'transformer_tokenizer1'
STATE_DICT_PATH = MODELS_DIR / 'model1.pt'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'transformers'

for directory in (RAW_DATA_DIR, PROCESSED_DATA_DIR, SPLIT_DATA_DIR, MODELS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print('???????????? ??????????:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('?????? ???????:', PROJECT_ROOT)


## ???????? ????????

????? ???????????? ???? ????????? CSV, ???? ?????? ?????? ?? CSV-????.


In [ ]:
DATASET_URL = ''  # ??? ????????????? ??????? ?????? ?????? ?? CSV
USE_UPLOADED_FILE = True  # True -> ???????? ????? ????? Colab, False -> ???????? ?? ??????

def load_dataset_from_source(dataset_url: str, use_uploaded_file: bool) -> tuple[pd.DataFrame, Path]:
    """????????? CSV ???? ?? ?????????? ?????, ???? ?? ?????? ??????."""
    if use_uploaded_file:
        uploaded_files = files.upload()
        if not uploaded_files:
            raise ValueError('???? ?? ??? ????????.')
        file_name = next(iter(uploaded_files))
        target_path = RAW_DATA_DIR / file_name
        target_path.write_bytes(uploaded_files[file_name])
    else:
        if not dataset_url.strip():
            raise ValueError('??? ???????? ?? ?????? ??????? DATASET_URL.')
        response = requests.get(dataset_url, timeout=60)
        response.raise_for_status()
        file_name = Path(dataset_url.split('?')[0]).name or 'dataset.csv'
        if not file_name.lower().endswith('.csv'):
            file_name += '.csv'
        target_path = RAW_DATA_DIR / file_name
        target_path.write_bytes(response.content)

    dataframe = pd.read_csv(target_path)
    return dataframe, target_path

raw_dataframe, raw_dataset_path = load_dataset_from_source(DATASET_URL, USE_UPLOADED_FILE)
print(raw_dataframe.head())
print('???? ???????? ?:', raw_dataset_path)


## ??????? ?????? ? ?????????? ?????????


In [ ]:
def find_column(columns: pd.Index, candidates: tuple[str, ...], description: str) -> str:
    """??????? ??????? ?? ?????? ?????????? ????."""
    normalized = {str(column).strip().casefold(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    raise ValueError(f'?? ??????? ??????? ??? {description}. ????????? ???????: {list(columns)}')

def clean_text(text: str) -> str:
    """????????? ??????? ??????? ?????? ???????."""
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text

def clean_label(label: str) -> str:
    """??????? ????? ??????."""
    return str(label).strip().lower()

text_column = find_column(raw_dataframe.columns, TEXT_COLUMN_CANDIDATES, '??????')
label_column = find_column(raw_dataframe.columns, LABEL_COLUMN_CANDIDATES, '????? ??????')

prepared_dataframe = raw_dataframe[[text_column, label_column]].copy()
prepared_dataframe.columns = ['text', 'label']
prepared_dataframe['text'] = prepared_dataframe['text'].fillna('').map(clean_text)
prepared_dataframe['label'] = prepared_dataframe['label'].fillna('').map(clean_label)
prepared_dataframe = prepared_dataframe[(prepared_dataframe['text'] != '') & (prepared_dataframe['label'] != '')]
prepared_dataframe = prepared_dataframe.drop_duplicates(subset=['text', 'label']).reset_index(drop=True)

if len(prepared_dataframe) < 12:
    raise ValueError('??? ???????? transformers-?????? ?????????? ????? ?? ????? 12 ?????????? ?????.')

processed_dataset_path = PROCESSED_DATA_DIR / 'news_transformer_processed.csv'
prepared_dataframe.to_csv(processed_dataset_path, index=False, encoding='utf-8')
print(prepared_dataframe['label'].value_counts())
print('????????? ??????? ???????? ?:', processed_dataset_path)


## ?????????? ?? train / validation / test


In [ ]:
train_dataframe, temp_dataframe = train_test_split(
    prepared_dataframe,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=prepared_dataframe['label'],
)
validation_dataframe, test_dataframe = train_test_split(
    temp_dataframe,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_dataframe['label'],
)

train_dataframe = train_dataframe.reset_index(drop=True)
validation_dataframe = validation_dataframe.reset_index(drop=True)
test_dataframe = test_dataframe.reset_index(drop=True)

train_path = SPLIT_DATA_DIR / 'train.csv'
validation_path = SPLIT_DATA_DIR / 'validation.csv'
test_path = SPLIT_DATA_DIR / 'test.csv'
train_dataframe.to_csv(train_path, index=False, encoding='utf-8')
validation_dataframe.to_csv(validation_path, index=False, encoding='utf-8')
test_dataframe.to_csv(test_path, index=False, encoding='utf-8')

print('Train:', len(train_dataframe), 'Validation:', len(validation_dataframe), 'Test:', len(test_dataframe))


## ?????????? ????? ? ???????????


In [ ]:
label_list = sorted(prepared_dataframe['label'].unique().tolist())
label_to_id = {label: index for index, label in enumerate(label_list)}
id_to_label = {index: label for label, index in label_to_id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(frame: pd.DataFrame) -> Dataset:
    dataset = Dataset.from_pandas(frame[['text', 'label']], preserve_index=False)
    dataset = dataset.map(lambda row: {'labels': label_to_id[row['label']]})
    return dataset

dataset_dict = DatasetDict({
    'train': to_dataset(train_dataframe),
    'validation': to_dataset(validation_dataframe),
    'test': to_dataset(test_dataframe),
})

def tokenize_batch(batch: dict[str, list[str]]) -> dict[str, Any]:
    return tokenizer(
        batch['text'],
        truncation=True,
        padding=False,
        max_length=256,
    )

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## ???????? ??????

??? ????????? ???????? ? Colab ????????????? ???????? GPU (`T4`).


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label={index: label for index, label in id_to_label.items()},
    label2id=label_to_id,
)

def compute_metrics(eval_prediction: tuple[np.ndarray, np.ndarray]) -> dict[str, float]:
    logits, labels = eval_prediction
    predictions = logits.argmax(axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': round(float(accuracy), 6),
        'precision_macro': round(float(precision), 6),
        'recall_macro': round(float(recall), 6),
        'f1_macro': round(float(f1_score), 6),
    }

training_arguments = TrainingArguments(
    output_dir=str(PROJECT_ROOT / 'trainer_output'),
    overwrite_output_dir=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_output = trainer.train()
validation_metrics = trainer.evaluate(tokenized_datasets['validation'])
test_metrics = trainer.evaluate(tokenized_datasets['test'], metric_key_prefix='test')
print('Validation metrics:', validation_metrics)
print('Test metrics:', test_metrics)


## ?????????? ??????????

???????????: `model1.pt`, ?????????? `save_pretrained`, ??????????? ? JSON-????? ?? ????????.


In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(TOKENIZER_DIR))
torch.save(model.state_dict(), STATE_DICT_PATH)

report_payload = {
    'generated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    'model_name': MODEL_NAME,
    'label_to_id': label_to_id,
    'id_to_label': {str(key): value for key, value in id_to_label.items()},
    'paths': {
        'raw_dataset_path': str(raw_dataset_path),
        'processed_dataset_path': str(processed_dataset_path),
        'train_path': str(train_path),
        'validation_path': str(validation_path),
        'test_path': str(test_path),
        'model_dir': str(MODEL_DIR),
        'tokenizer_dir': str(TOKENIZER_DIR),
        'state_dict_path': str(STATE_DICT_PATH),
    },
    'train_runtime_seconds': round(float(train_output.metrics.get('train_runtime', 0.0)), 4),
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
}

report_path = REPORTS_DIR / 'transformer_training_report.json'
report_path.write_text(json.dumps(report_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('??????????? ?????????:')
print(' - model directory:', MODEL_DIR)
print(' - tokenizer directory:', TOKENIZER_DIR)
print(' - state dict:', STATE_DICT_PATH)
print(' - report:', report_path)


## ????

????? ?????????? ???????? ? ??????? ????? ?????????:

- ???????? ? ????????? ????????;
- `train / validation / test`;
- ?????????? ???????????? ?????? `save_pretrained`;
- ???????????;
- ???? ????? `model1.pt`;
- JSON-????? ? ????????? ????????.
